# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matiasdibe23-sys/Flyrank-Starter-Notebooks/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Archetype & Reason Code Mapping

We translate observed traffic dynamics and performance scores into decision-support categories:

1. **HIGH_RISK_REFRESH_PRIORITY (`REASON_CODE: DECAY_MOMENTUM_HIGH_IMPACT`)**
   * **Criteria:** High $H_1$ traffic ($\ge 50$ clicks) AND `click_momentum` $< 0.85$.
   * **Recommended Action:** Immediate editorial refresh (update facts, improve search intent alignment, refresh outdated sections).

2. **MONITOR_POSITION_SLIP (`REASON_CODE: POSITION_DRIFT`)**
   * **Criteria:** Average position dropped into the $10-20$ range while impressions remain high.
   * **Recommended Action:** On-page SEO optimization (refactor title tags, meta descriptions, and internal linking).

3. **STABLE_HIGH_PERFORMER (`REASON_CODE: MAINTAIN_STATUS_QUO`)**
   * **Criteria:** High clicks AND `click_momentum` $\ge 0.95$.
   * **Recommended Action:** Do not modify (protect existing search rank and prevent unintended traffic loss).

In [ ]:
import duckdb
import numpy as np
import pandas as pd

# 1. Connect and query the feature and label source windows
PANEL_URL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet"
con = duckdb.connect()
if PANEL_URL.startswith("hf://"):
    con.execute("INSTALL httpfs; LOAD httpfs;")
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
        con.execute(f"CREATE SECRET hf_auth (TYPE HTTP, BEARER_TOKEN '{hf_token}');")
    except Exception:
        pass

raw_df = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-06-15') AS impr_h1,
        SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-06-15') AS clicks_h1,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date <= DATE '2026-06-15')
            / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-06-15'), 0) AS pos_h1,
        SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-06-07') AS clicks_w1,
        SUM(gsc_clicks) FILTER (WHERE report_date BETWEEN DATE '2026-06-08' AND DATE '2026-06-15') AS clicks_w2
    FROM read_parquet('{PANEL_URL}')
    WHERE month = '2026-06' AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-06-15') >= 20
""").df()

# Compute features
rate_w1 = raw_df.clicks_w1.fillna(0) / 7.0
rate_w2 = raw_df.clicks_w2.fillna(0) / 8.0
raw_df["click_momentum"] = ((rate_w2 + 1.0) / (rate_w1 + 1.0)).fillna(1.0)

# Assign Archetype, Reason Code, and Recommended Action
def map_action(row):
    if row["clicks_h1"] >= 50 and row["click_momentum"] < 0.85:
        return "HIGH_RISK_REFRESH_PRIORITY", "DECAY_MOMENTUM_HIGH_IMPACT", "Urgent Editorial Refresh"
    elif row["pos_h1"] > 10.0 and row["click_momentum"] < 0.95:
        return "MONITOR_POSITION_SLIP", "POSITION_DRIFT", "On-Page SEO Optimization"
    else:
        return "STABLE_HIGH_PERFORMER", "MAINTAIN_STATUS_QUO", "No Action Needed"

results = raw_df.apply(map_action, axis=1)
raw_df["archetype"] = [r[0] for r in results]
raw_df["reason_code"] = [r[1] for r in results]
raw_df["recommended_action"] = [r[2] for r in results]

# Rank queue by expected click loss
raw_df["expected_loss_score"] = raw_df["clicks_h1"] * (1.0 - raw_df["click_momentum"].clip(upper=1.0))
playbook_queue = raw_df.sort_values("expected_loss_score", ascending=False).reset_index(drop=True)

print("=== CONTENT ACTION QUEUE (TOP 10) ===")
print(playbook_queue[["content_hash_id", "clicks_h1", "click_momentum", "archetype", "reason_code", "recommended_action"]].head(10).to_string(index=False))

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use & Known Limitations

* **Intended Use:** Internal decision-support system designed to assist content marketing and SEO teams in prioritizing manual content refreshes and optimization workflows.
* **Known Limits:**
  * Evaluated exclusively on a 30-day panel (June 2026 data).
  * Does not account for seasonal traffic fluctuations, sitewide migrations, or Google core updates.
  * Observed click momentum serves as a directional signal, not a guaranteed prediction of post-refresh traffic gains.

In [ ]:
# Section 2 Code Verification: Distribution of recommendations across the queue
summary_use = playbook_queue["archetype"].value_counts(normalize=True).rename("Share").to_frame()
summary_use["Count"] = playbook_queue["archetype"].value_counts()
print("=== QUEUE ARCHETYPE DISTRIBUTION ===")
print(summary_use.to_string())

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Protocols & The No-Go List

1. **Human-in-the-Loop Rules:**
   * **High-Volume Threshold:** Any page generating $>500$ monthly clicks requires explicit approval from a Senior Content Strategist prior to making edits.
   * **Intent Verification:** Content managers must manually verify search intent for pages flagged under `POSITION_DRIFT` before altering title tags or headers.

2. **The No-Go List (STRICTLY DO NOT AUTOMATE):**
   * Core brand landing pages and homepage assets.
   * Legal terms, privacy policy, and compliance documentation.
   * High-converting transactional product or sales pages.

In [ ]:
# Section 3 Code Verification: Identifying high-volume pages requiring human review
high_volume_flagged = playbook_queue[(playbook_queue["clicks_h1"] > 500) & (playbook_queue["archetype"] == "HIGH_RISK_REFRESH_PRIORITY")]
print(f"Total high-risk pages requiring Senior Review (>500 clicks): {len(high_volume_flagged)}")

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring & Retrain Triggers

1. **Monitoring Metrics:**
   * Track bi-weekly distribution shifts in `click_momentum` across key client accounts.
   * Measure the conversion rate of flagged recommendations into completed editorial reviews.

2. **Retrain / Recalibration Triggers:**
   * Trigger model recalibration when multi-month historical data including explicit post-refresh outcome events becomes available.
   * Recalibrate if the global average decay rate deviates by more than $15\%$ from baseline expectations.

In [ ]:
# Section 4 Code Verification: Simulated monitoring metric (Momentum Drift Check)
baseline_momentum_mean = playbook_queue["click_momentum"].mean()
print(f"Baseline Mean Click Momentum: {baseline_momentum_mean:.4f}")
print("[OK] Monitoring baseline established for tracking future data drift.")

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for Research Paper Integration

We export the generated action queue to `work/outputs/w07_content_action_queue.csv` and summary figures to `work/figures/` for downstream inclusion in the final research paper.

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1. Export Action Queue CSV
queue_export_path = "work/outputs/w07_content_action_queue.csv"
playbook_queue.to_csv(queue_export_path, index=False)
print(f"[EXPORTS] Action queue exported successfully to {queue_export_path} ({len(playbook_queue):,} rows).")

# 2. Generate and export Archetype Distribution Chart
plt.figure(figsize=(8, 4))
playbook_queue["archetype"].value_counts().plot(kind="bar", color="#2563EB")
plt.title("Content Action Playbook — Archetype Distribution")
plt.xlabel("Archetype Category")
plt.ylabel("Number of Pages")
plt.tight_layout()

figure_path = "work/figures/w07_archetype_distribution.png"
plt.savefig(figure_path, dpi=300)
plt.close()

print(f"[EXPORTS] Summary figure saved to {figure_path}.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.